# Final Pipeline: RetinaNet + RT-DETR-L (Kaggle)

**Pipeline:** дві окремі моделі → єдина official validation COCO evaluation → порівняння RetinaNet, RT-DETR, TTA, NMS-ensemble та Weighted Boxes Fusion → `submission.csv`.

RetinaNet ResNet50-FPN (`LR=5e-5`, class ids 1–8, ignored-region loss), RT-DETR-L Ultralytics (pretrained `rtdetr-l.pt`, 50 denoising queries).

## 1. Конфігурація та залежності

In [10]:
from pathlib import Path

pt_files = sorted(
    Path("/kaggle/input").rglob("*.pt")
)

print("Found checkpoints:", len(pt_files))

for path in pt_files:
    print(
        path,
        f"({path.stat().st_size / 1024**2:.1f} MB)"
    )

Found checkpoints: 2
/kaggle/input/datasets/nastyadovbenchuk/retinanet/retinanet_full_s3_e10_cosine_lr5e-5/best_model.pt (123.8 MB)
/kaggle/input/datasets/nastyadovbenchuk/retinanet/retinanet_full_s3_e10_cosine_lr5e-5/last.pt (368.7 MB)


In [ ]:
import os, sys, subprocess, importlib.util, json, random, re, math, time, io, contextlib, hashlib, gc, gzip, shutil, copy
from pathlib import Path
from collections import defaultdict
for module, package in [('pycocotools','pycocotools'), ('yaml','pyyaml'),
                        ('ultralytics','ultralytics'), ('ensemble_boxes','ensemble-boxes')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
import torch, torchvision, numpy as np, pandas as pd, yaml, ultralytics
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from torchvision.ops import batched_nms, box_iou
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics import RTDETR
from ensemble_boxes import weighted_boxes_fusion

assert torch.cuda.is_available(), 'Enable GPU in Kaggle Settings.'
DEVICE=torch.device('cuda')
SEED=42
TRAIN_FRAME_STEP=3
VAL_FRAME_STEP=2
FINAL_VAL_FRAME_STEP=1
BATCH_SIZE=2
NUM_WORKERS=2
RETINA_EPOCHS=10
RETINA_LR=5e-5
RETINA_WEIGHT_DECAY=0.01
RETINA_MIN_SIZE=800
RETINA_MAX_SIZE=1333
RETINA_AMP=True
RETINA_VAL_EVERY=2
RETINA_TTA_ABLATION=True
RTDETR_EPOCHS=10
RTDETR_BATCH=2
RTDETR_IMGSZ=640
RTDETR_LR=0.001
RTDETR_DENOISING=50
RTDETR_AMP=True
EVAL_CONF=0.02
MAX_DETECTIONS=500
COCO_MAX_DETS=100

# Explicit overrides prevent ambiguous automatic discovery.
DATA_ROOT_OVERRIDE=None
PREP_ROOT_OVERRIDE=None
TEST_META_ROOT_OVERRIDE=None
RETINA_BEST_INPUT='/kaggle/input/datasets/nastyadovbenchuk/retinanet/retinanet_full_s3_e10_cosine_lr5e-5/best_model.pt'
RTDETR_BEST_INPUT=None
RTDETR_PRETRAINED='rtdetr-l.pt'
RETINA_FORCE_TRAIN=False
RTDETR_TRAIN=True
RUN_TEST=True
RUN_SMOKE_TEST=True

COMPETITION_SLUG='multiple-object-video-detection-hw-2-ucu-ai-2026'
INPUT_ROOT=Path('/kaggle/input')
OUT=Path('/kaggle/working/final_retina_rtdetr_pipeline')
OUT.mkdir(parents=True,exist_ok=True)
RUN_NAME='final_retina_rtdetr_ensemble'
CLASS_NAMES={1:'person',2:'bicycle',3:'car',4:'van',5:'truck',6:'tricycle',7:'bus',8:'motor'}
NAMES=[CLASS_NAMES[i] for i in range(1,9)]
NUM_CLASSES=9
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print('GPU:',torch.cuda.get_device_name(0))
print('Torch/TorchVision:',torch.__version__,torchvision.__version__)
print('Ultralytics:',ultralytics.__version__)
print('Output:',OUT)
EARLY_CONFIG={
    'seed':SEED,'train_step':TRAIN_FRAME_STEP,'val_step':VAL_FRAME_STEP,
    'final_val_step':FINAL_VAL_FRAME_STEP,'retina_lr':RETINA_LR,'retina_epochs':RETINA_EPOCHS,
    'retina_img':[RETINA_MIN_SIZE,RETINA_MAX_SIZE],'rtdetr_epochs':RTDETR_EPOCHS,
    'rtdetr_imgsz':RTDETR_IMGSZ,'rtdetr_lr':RTDETR_LR,'rtdetr_warmup_epochs':3.0,
    'rtdetr_denoising':RTDETR_DENOISING,'eval_conf':EVAL_CONF,
    'retina_tta':RETINA_TTA_ABLATION,'retina_best_input':str(RETINA_BEST_INPUT),
    'rtdetr_best_input':str(RTDETR_BEST_INPUT)}
contract=OUT/'run_contract.json'
if contract.is_file():
    previous=json.loads(contract.read_text())
    assert previous==EARLY_CONFIG, 'Cached runs with a different config are unsafe. Use a new OUT directory or explicitly clear old predictions.'
else:
    contract.write_text(json.dumps(EARLY_CONFIG,indent=2))

GPU: Tesla T4
Torch/TorchVision: 2.11.0+cu130 0.26.0+cu130
Ultralytics: 8.4.204
Output: /kaggle/working/final_retina_rtdetr_pipeline


## 2. Data discovery і submission preflight

Перевіряємо відповідність ID та наявність зображень.

In [ ]:
def discover_data_root():
    if DATA_ROOT_OVERRIDE:
        p=Path(DATA_ROOT_OVERRIDE)
        assert (p/'train/annotations_coco.json').exists() and (p/'val/annotations_coco.json').exists(),p
        return p
    candidates=[INPUT_ROOT/'competitions'/COMPETITION_SLUG,INPUT_ROOT/COMPETITION_SLUG]
    candidates += [f.parent.parent for f in INPUT_ROOT.rglob('annotations_coco.json') if f.parent.name=='train']
    matches=list(dict.fromkeys(p for p in candidates
        if (p/'train/annotations_coco.json').is_file() and (p/'val/annotations_coco.json').is_file()))
    if len(matches)!=1:
        raise FileNotFoundError(f'Expected one train/val root, got {matches}. Set DATA_ROOT_OVERRIDE.')
    return matches[0]

def discover_prep():
    if PREP_ROOT_OVERRIDE:
        p=Path(PREP_ROOT_OVERRIDE)
        assert all((p/f).is_file() for f in ('train.json','val.json')),p
        return p
    matches=list(dict.fromkeys(f.parent for f in INPUT_ROOT.rglob('train.json')
                                  if (f.parent/'val.json').is_file()))
    if len(matches)==1:return matches[0]
    if matches:print('Multiple preprocessing roots; using raw COCO.',matches)
    return None

def discover_test_meta():
    if TEST_META_ROOT_OVERRIDE:
        p=Path(TEST_META_ROOT_OVERRIDE)
        assert (p/'test_images.csv').is_file(),p
        return p
    candidates=[DATA_ROOT, INPUT_ROOT/'competitions'/COMPETITION_SLUG,INPUT_ROOT/COMPETITION_SLUG]
    candidates += [f.parent for f in INPUT_ROOT.rglob('test_images.csv')]
    candidates=list(dict.fromkeys(p for p in candidates if (p/'test_images.csv').is_file()))
    if not candidates: raise FileNotFoundError('No test_images.csv. Add competition Input or set TEST_META_ROOT_OVERRIDE.')
    if DATA_ROOT in candidates:return DATA_ROOT
    if len(candidates)!=1:raise ValueError(f'Multiple metadata roots: {candidates}. Set TEST_META_ROOT_OVERRIDE.')
    return candidates[0]

DATA_ROOT=discover_data_root(); PREP_ROOT=discover_prep(); TEST_META_ROOT=discover_test_meta()
print('DATA_ROOT =',DATA_ROOT)
print('PREP_ROOT =',PREP_ROOT)
print('TEST_META_ROOT =',TEST_META_ROOT)
print('sample_submission:',(TEST_META_ROOT/'sample_submission.csv').is_file())
test_info=pd.read_csv(TEST_META_ROOT/'test_images.csv',dtype={'image_id':str})
assert {'image_id','file_name','width','height'}.issubset(test_info.columns),test_info.columns.tolist()
assert test_info.image_id.notna().all() and test_info.image_id.is_unique
assert not test_info.file_name.isna().any()
assert (test_info[['width','height']].astype(float)>0).all().all()
if (TEST_META_ROOT/'sample_submission.csv').is_file():
    sample=pd.read_csv(TEST_META_ROOT/'sample_submission.csv',dtype={'image_id':str},keep_default_na=False)
    assert {'image_id','PredictionString'}.issubset(sample.columns)
    assert set(sample.image_id)==set(test_info.image_id)
    sample=sample[['image_id','PredictionString']]
else:
    sample=pd.DataFrame({'image_id':test_info.image_id.copy(),
                         'PredictionString':['']*len(test_info)})
assert sample.image_id.is_unique

def relative_safe(name):
    p=Path(str(name).replace('\\','/').lstrip('/'))
    assert '..' not in p.parts, f'Unsafe image path {name}'
    return p

def locate_image(split,name,override_root=None):
    rel=relative_safe(name)
    roots=[Path(override_root)] if override_root is not None else [DATA_ROOT,TEST_META_ROOT]
    for root in dict.fromkeys(roots):
        candidates=[root/rel,root/split/rel,root/split/'images'/rel]
        if rel.parts and rel.parts[0]==split:
            tail=Path(*rel.parts[1:]);candidates.extend([root/split/tail,root/tail])
        for p in candidates:
            if p.is_file():return p
    raise FileNotFoundError(f'Cannot find {split} frame {name} under {roots}')

for idx in sorted(set([0,len(test_info)//2,len(test_info)-1])):
    row=test_info.iloc[idx]
    print('Test sample:',row.image_id,locate_image('test',row.file_name))
print('Test frames:',len(test_info))

DATA_ROOT = /kaggle/input/datasets/nastyadovbenchuk/multiple-object-video-detection
PREP_ROOT = /kaggle/input/datasets/nastyadovbenchuk/preprocessed-result
TEST_META_ROOT = /kaggle/input/datasets/nastyadovbenchuk/multiple-object-video-detection
sample_submission: False
Test sample: clip_001_0000001 /kaggle/input/datasets/nastyadovbenchuk/multiple-object-video-detection/test/images/clip_001/0000001.jpg
Test sample: clip_025_0000083 /kaggle/input/datasets/nastyadovbenchuk/multiple-object-video-detection/test/images/clip_025/0000083.jpg
Test sample: clip_048_0000045 /kaggle/input/datasets/nastyadovbenchuk/multiple-object-video-detection/test/images/clip_048/0000045.jpg
Test frames: 3811


## 3. Відбір кадрів за за відеопослідовностями

In [ ]:
def canonical_rel(split,raw_name):
    p=locate_image(split,raw_name)
    return p.relative_to(DATA_ROOT/split).as_posix()

def read_coco_split(split):
    doc=json.loads((DATA_ROOT/split/'annotations_coco.json').read_text())
    anns_by_img=defaultdict(list)
    for a in doc['annotations']:anns_by_img[a['image_id']].append(a)
    rows=[]
    for im in doc['images']:
        w,h=int(im['width']),int(im['height'])
        boxes,labels,ignored=[],[],[]
        for a in anns_by_img[im['id']]:
            x,y,bw,bh=map(float,a['bbox'])
            b=[max(0.,x),max(0.,y),min(float(w),x+bw),min(float(h),y+bh)]
            if b[2]<=b[0] or b[3]<=b[1]:continue
            cls=int(a.get('category_id',-1))
            if a.get('iscrowd',0) or cls not in CLASS_NAMES:
                ignored.append(b)
            else:
                boxes.append(b);labels.append(cls)
        rows.append({'image_id':int(im['id']), 'file_name':canonical_rel(split,im['file_name']),
                     'width':w,'height':h,'boxes':boxes,'labels':labels,'ignored_boxes':ignored})
    return rows

def load_split(split):
    source=read_coco_split(split)
    if PREP_ROOT is None:return source
    rows=json.loads((PREP_ROOT/f'{split}.json').read_text())
    by_id={int(x['image_id']):x for x in source}
    assert len(rows)==len(source) and len({int(x['image_id']) for x in rows})==len(source)
    normalized=[]
    for r in rows:
        meta=by_id[int(r['image_id'])]
        assert set(map(int,r['labels'])).issubset(CLASS_NAMES)
        normalized.append({**meta,'boxes':r['boxes'],'labels':r['labels'],
                           'ignored_boxes':r.get('ignored_boxes',[])})
    return normalized

train_records=load_split('train');val_records=load_split('val')
assert len(train_records)==24201 and len(val_records)==2846,(len(train_records),len(val_records))
for part in (train_records,val_records):
    assert len({int(r['image_id']) for r in part})==len(part)
    for r in part:assert len(r['boxes'])==len(r['labels'])

def seq_name(rec):
    return Path(rec['file_name']).parent.name

def frame_key(rec):
    stem=Path(rec['file_name']).stem
    digits=re.search(r'(\d+)$',stem)
    return (int(digits.group(1)) if digits else -1,stem)

def subsample_by_seq(records,step):
    groups=defaultdict(list)
    for r in records:groups[seq_name(r)].append(r)
    return [r for s in sorted(groups) for r in sorted(groups[s],key=frame_key)[::step]]

assert {seq_name(r) for r in train_records}.isdisjoint({seq_name(r) for r in val_records}), 'Sequence leakage!'
train_selected=subsample_by_seq(train_records,TRAIN_FRAME_STEP)
val_selected=subsample_by_seq(val_records,VAL_FRAME_STEP)
val_final=subsample_by_seq(val_records,FINAL_VAL_FRAME_STEP)
for name,rows in [('train',train_selected),('val_for_checkpoints',val_selected),('val_final',val_final)]:
    pd.DataFrame([{'image_id':r['image_id'],'file_name':r['file_name'],'sequence':seq_name(r)}
        for r in rows]).to_csv(OUT/f'{name}_frames.csv',index=False)
print('Train',len(train_selected),'/ 24201 | checkpoint val',len(val_selected),
      '| final eval val',len(val_final))

Train 8090 / 24201 | checkpoint val 1425 | final eval val 2846


## 4. COCO evaluation

Однакові GT, ignored/crowd areas, `maxDets=100` та об'єднання по `image_id` для обох моделей. 

In [ ]:
def xyxy_to_xywh(b):
    x1,y1,x2,y2=map(float,b)
    return [x1,y1,max(0.,x2-x1),max(0.,y2-y1)]

def xywh_to_xyxy(b):
    x,y,w,h=map(float,b)
    return [x,y,x+w,y+h]

def make_coco_gt(records):
    imgs=[];anns=[]
    for r in records:
        iid=int(r['image_id'])
        imgs.append({'id':iid,'width':int(r['width']),'height':int(r['height']),
                     'file_name':r['file_name']})
        for b,l in zip(r['boxes'],r['labels']):
            a=xyxy_to_xywh(b)
            if a[2]<=0 or a[3]<=0:continue
            anns.append({'id':len(anns)+1,'image_id':iid,'category_id':int(l),
                         'bbox':a,'area':a[2]*a[3],'iscrowd':0})
        for b in r.get('ignored_boxes',[]):
            a=xyxy_to_xywh(b)
            if a[2]<=0 or a[3]<=0:continue
            for cls in CLASS_NAMES:
                anns.append({'id':len(anns)+1,'image_id':iid,'category_id':cls,
                             'bbox':a,'area':a[2]*a[3],'iscrowd':1,'ignore':1})
    gt=COCO()
    gt.dataset={'images':imgs,'annotations':anns,'categories':[
        {'id':i,'name':n} for i,n in CLASS_NAMES.items()], 'info':{},'licenses':[]}
    with contextlib.redirect_stdout(io.StringIO()):gt.createIndex()
    return gt

COCO_FINAL=make_coco_gt(val_final)
COCO_CHECKPOINT=make_coco_gt(val_selected)

def coco_metrics(preds,coco_gt=COCO_FINAL):
    with contextlib.redirect_stdout(io.StringIO()):
        if preds:
            coco_dt=coco_gt.loadRes(preds)
        else:
            coco_dt=COCO();coco_dt.dataset={'images':coco_gt.dataset['images'],
                'annotations':[],'categories':coco_gt.dataset['categories'], 'info':{},'licenses':[]}
            coco_dt.createIndex()
        ev=COCOeval(coco_gt,coco_dt,'bbox')
        ev.params.imgIds=[int(i['id']) for i in coco_gt.dataset['images']]
        ev.params.catIds=list(CLASS_NAMES)
        ev.params.maxDets=[1,10,COCO_MAX_DETS]
        ev.evaluate();ev.accumulate();ev.summarize()
    s=ev.stats
    return {'mAP':float(s[0]),'AP50':float(s[1]),'AP75':float(s[2]),
            'AP_small':float(s[3]),'AP_medium':float(s[4]),'AP_large':float(s[5])}

def dump_gz(path,data):
    with gzip.open(path,'wt',encoding='utf8') as f:json.dump(data,f,separators=(',',':'))

def read_gz(path):
    with gzip.open(path,'rt',encoding='utf8') as f:return json.load(f)

def pred_from_xyxy(image_id,boxes,labels,scores):
    out=[]
    for b,l,sc in zip(boxes,labels,scores):
        if int(l) not in CLASS_NAMES:continue
        bb=xyxy_to_xywh(b)
        if bb[2]>0 and bb[3]>0 and float(sc)>=EVAL_CONF:
            out.append({'image_id':image_id if isinstance(image_id,str) else int(image_id),'category_id':int(l),
                        'bbox':bb,'score':float(sc)})
    return out

def group_predictions(rows):
    d=defaultdict(list)
    for x in rows:d[str(x['image_id'])].append(x)
    return d

(OUT/'class_mapping.json').write_text(json.dumps({'retina_coco':CLASS_NAMES,
    'rtdetr_to_coco':{str(i-1):i for i in CLASS_NAMES},'kaggle_from_coco':'category_id - 1'},indent=2))

332

## 5. RetinaNet (loading previous checkpoint або train once)

In [ ]:
retina_matches=[]
if RETINA_BEST_INPUT is not None:
    retina_matches=[Path(RETINA_BEST_INPUT)]
else:
    retina_matches=list(INPUT_ROOT.rglob('best_model.pt'))
    if len(retina_matches)>1:
        raise ValueError(f'Multiple RetinaNet best_model.pt found: {retina_matches}. Set RETINA_BEST_INPUT explicitly.')
RETINA_EXTERNAL_BEST=retina_matches[0] if retina_matches else None
if RETINA_EXTERNAL_BEST is not None:assert RETINA_EXTERNAL_BEST.is_file()
RETINA_TRAIN=RETINA_FORCE_TRAIN or (RETINA_EXTERNAL_BEST is None and not (OUT/'retina_best.pt').is_file())
print('RetinaNet:', 'TRAIN full train' if RETINA_TRAIN else 'LOAD existing weights')
print('Existing checkpoint:',RETINA_EXTERNAL_BEST)

class RetinaDataset(Dataset):
    def __init__(self,records,split='train',augment=False):
        self.rows=list(records);self.split=split;self.augment=augment
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        r=self.rows[i]
        with Image.open(DATA_ROOT/self.split/r['file_name']) as im:
            rgb=np.asarray(im.convert('RGB')).copy()
        h,w=rgb.shape[:2]
        img=torch.from_numpy(rgb).permute(2,0,1).float()/255
        boxes=torch.tensor(r['boxes'],dtype=torch.float32).reshape(-1,4)
        ignored=torch.tensor(r.get('ignored_boxes',[]),dtype=torch.float32).reshape(-1,4)
        labels=torch.tensor(r['labels'],dtype=torch.int64)
        if self.augment and random.random()<0.5:
            img=img.flip(-1)
            for bb in (boxes,ignored):
                if len(bb):
                    oldl=bb[:,0].clone();oldr=bb[:,2].clone()
                    bb[:,0]=w-oldr;bb[:,2]=w-oldl
        return img,{'boxes':boxes,'labels':labels,'ignored_boxes':ignored,
                    'image_id':torch.tensor(int(r['image_id']))}

def collate(batch):
    imgs,targets=zip(*batch);return list(imgs),list(targets)

def worker_seed(worker_id):
    s=torch.initial_seed()%2**32
    random.seed(s);np.random.seed(s)

from types import MethodType
from torchvision.models.detection import retinanet_resnet50_fpn,RetinaNet_ResNet50_FPN_Weights
from torchvision.models.detection.retinanet import RetinaNetClassificationHead
from torchvision.models.detection.transform import resize_boxes
from torchvision.models.detection import _utils as det_utils

def make_retina_model(with_pretraining=False):
    net=retinanet_resnet50_fpn(
        weights=RetinaNet_ResNet50_FPN_Weights.DEFAULT if with_pretraining else None,
        weights_backbone=None,min_size=RETINA_MIN_SIZE,max_size=RETINA_MAX_SIZE,
        score_thresh=EVAL_CONF,detections_per_img=MAX_DETECTIONS)
    anchors=net.anchor_generator.num_anchors_per_location()[0]
    net.head.classification_head=RetinaNetClassificationHead(net.backbone.out_channels,anchors,NUM_CLASSES)
    original_resize=net.transform.resize
    def resize_with_ignore(img,target=None):
        orig=img.shape[-2:]
        img,target=original_resize(img,target)
        if target is not None and 'ignored_boxes' in target:
            target['ignored_boxes']=resize_boxes(target['ignored_boxes'],orig,img.shape[-2:])
        return img,target
    net.transform.resize=resize_with_ignore
    def compute_loss_with_ignore(self,targets,head_outputs,anchors):
        matches=[]
        for anchor,target in zip(anchors,targets):
            gt=target['boxes']
            matched=self.proposal_matcher(box_iou(gt,anchor)) if len(gt) else torch.full(
                (len(anchor),),-1,device=anchor.device,dtype=torch.long)
            ignored=target.get('ignored_boxes')
            if ignored is not None and len(ignored):
                centers=(anchor[:,:2]+anchor[:,2:])/2
                mask=torch.zeros(len(anchor),dtype=torch.bool,device=anchor.device)
                for region in ignored.unique(dim=0):
                    mask|=((centers[:,0]>=region[0])&(centers[:,0]<=region[2])&
                           (centers[:,1]>=region[1])&(centers[:,1]<=region[3]))
                matched[(matched<0)&mask]=det_utils.Matcher.BETWEEN_THRESHOLDS
            matches.append(matched)
        return self.head.compute_loss(targets,head_outputs,anchors,matches)
    net.compute_loss=MethodType(compute_loss_with_ignore,net)
    return net.to(DEVICE)

RET_BEST=OUT/'retina_best.pt';RET_LAST=OUT/'retina_last.pt'
if RETINA_EXTERNAL_BEST is not None and not RETINA_FORCE_TRAIN and not RET_BEST.exists():
    shutil.copy2(RETINA_EXTERNAL_BEST,RET_BEST)

# Check that pretrained RetinaNet checkpoint really has the expected shape/mapping.
def load_retina_weights(net,path):
    ck=torch.load(str(path),map_location='cpu',weights_only=True)
    state=ck['model'] if isinstance(ck,dict) and 'model' in ck else ck
    net.load_state_dict(state,strict=True)
    return ck

def atomic_save(payload,path):
    tmp=Path(str(path)+'.tmp')
    torch.save(payload,tmp);tmp.replace(path)

if not RETINA_TRAIN and not RET_BEST.exists():
    raise FileNotFoundError('RetinaNet best checkpoint missing. Attach input or set RETINA_FORCE_TRAIN=True.')

RetinaNet: LOAD existing weights
Existing checkpoint: /kaggle/input/datasets/nastyadovbenchuk/retinanet/retinanet_full_s3_e10_cosine_lr5e-5/best_model.pt


## 6. RetinaNet training (тільки якщо checkpoint відсутній)

Навчаємо `RetinaNet` із ignored-anchor masking, AdamW, cosine LR, AMP і перевіркою mAP через кожні 2 епохи. Зберігаємо best/last, optimizer/scaler та історію.

In [20]:
def retina_clean(raw,w,h,thr=EVAL_CONF):
    boxes=raw['boxes'].detach().float().cpu().reshape(-1,4).clone()
    scores=raw['scores'].detach().float().cpu().reshape(-1)
    labels=raw['labels'].detach().long().cpu().reshape(-1)
    boxes[:,[0,2]]=boxes[:,[0,2]].clamp(0,w)
    boxes[:,[1,3]]=boxes[:,[1,3]].clamp(0,h)
    m=(torch.isfinite(boxes).all(1)&torch.isfinite(scores)&(scores>=thr)&
       (boxes[:,2]>boxes[:,0])&(boxes[:,3]>boxes[:,1])&(labels>=1)&(labels<=8))
    return {'boxes':boxes[m],'scores':scores[m],'labels':labels[m]}

def retina_batch_predict(net,images):
    net.eval()
    with torch.inference_mode(),torch.autocast(device_type='cuda',enabled=RETINA_AMP):
        return net([x.to(DEVICE,non_blocking=True) for x in images])

def retina_infer_one(net,image,tta=False,base=None):
    h,w=image.shape[-2:]
    if base is None:
        base=retina_clean(retina_batch_predict(net,[image])[0],w,h)
    if not tta:return base
    flip=retina_clean(retina_batch_predict(net,[image.flip(-1)])[0],w,h)
    if len(flip['boxes']):
        b=flip['boxes']; x1=b[:,0].clone();x2=b[:,2].clone()
        b[:,0]=w-x2;b[:,2]=w-x1
    b=torch.cat([base['boxes'],flip['boxes']]);s=torch.cat([base['scores'],flip['scores']]);l=torch.cat([base['labels'],flip['labels']])
    if len(b):
        keep=batched_nms(b,s,l,0.55)[:MAX_DETECTIONS]
        b,s,l=b[keep],s[keep],l[keep]
    return {'boxes':b,'scores':s,'labels':l}

def retina_val_preds(net,records,tta=False):
    rows=[];elapsed=[]
    net.eval()
    for r in tqdm(records,desc='Retina val TTA' if tta else 'Retina val',leave=False):
        with Image.open(DATA_ROOT/'val'/r['file_name']) as im:
            rgb=np.asarray(im.convert('RGB')).copy()
        h,w=rgb.shape[:2]
        img=torch.from_numpy(rgb).permute(2,0,1).float()/255
        torch.cuda.synchronize();t=time.perf_counter()
        output=retina_infer_one(net,img,tta=tta)
        torch.cuda.synchronize();elapsed.append(time.perf_counter()-t)
        rows+=pred_from_xyxy(r['image_id'],output['boxes'].tolist(),
                             output['labels'].tolist(),output['scores'].tolist())
    return rows,1000*np.mean(elapsed)

if RETINA_TRAIN:
    net=make_retina_model(with_pretraining=not RET_LAST.exists())
    if RUN_SMOKE_TEST:
        sample_loader=DataLoader(RetinaDataset(train_selected[:2],augment=True),batch_size=2,collate_fn=collate)
        xs,ys=next(iter(sample_loader))
        xs=[x.to(DEVICE) for x in xs];ys=[{k:v.to(DEVICE) for k,v in y.items()} for y in ys]
        net.train()
        with torch.no_grad(),torch.autocast('cuda',enabled=RETINA_AMP): parts=net(xs,ys)
        assert all(torch.isfinite(x) for x in parts.values())
        print('Retina smoke loss:',{k:float(v) for k,v in parts.items()})
        del xs,ys,parts
    loader=DataLoader(RetinaDataset(train_selected,augment=True),batch_size=BATCH_SIZE,shuffle=True,
        num_workers=NUM_WORKERS,collate_fn=collate,pin_memory=True,
        worker_init_fn=worker_seed,persistent_workers=NUM_WORKERS>0)
    opt=torch.optim.AdamW(net.parameters(),lr=RETINA_LR,weight_decay=RETINA_WEIGHT_DECAY)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=RETINA_EPOCHS,eta_min=1e-5)
    scaler=torch.amp.GradScaler('cuda',enabled=RETINA_AMP)
    best_map=-1.;start_epoch=0;history=[]
    signature={'train_step':TRAIN_FRAME_STEP,'val_step':VAL_FRAME_STEP,'epochs':RETINA_EPOCHS,
               'batch_size':BATCH_SIZE,'lr':RETINA_LR,'seed':SEED,'img_min':RETINA_MIN_SIZE,
               'img_max':RETINA_MAX_SIZE,'weight_decay':RETINA_WEIGHT_DECAY}
    hash_sig=hashlib.sha256(json.dumps(signature,sort_keys=True).encode()).hexdigest()
    if RET_LAST.exists():
        old=torch.load(RET_LAST,map_location='cpu',weights_only=True)
        assert old['config_hash']==hash_sig,'Last checkpoint config changed; do not silently resume.'
        net.load_state_dict(old['model']);opt.load_state_dict(old['optimizer'])
        sch.load_state_dict(old['scheduler']);scaler.load_state_dict(old['scaler'])
        best_map=float(old['best_map']);start_epoch=int(old['epoch']);history=old['history']
        print('Resumed Retina epoch',start_epoch)
    for epoch in range(start_epoch+1,RETINA_EPOCHS+1):
        net.train();sums=defaultdict(float);t0=time.perf_counter();n=0
        for imgs,tgts in tqdm(loader,desc=f'Retina epoch {epoch}/{RETINA_EPOCHS}',leave=False):
            imgs=[x.to(DEVICE,non_blocking=True) for x in imgs]
            tgts=[{k:v.to(DEVICE,non_blocking=True) for k,v in t.items()} for t in tgts]
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda',enabled=RETINA_AMP):
                parts=net(imgs,tgts);loss=sum(parts.values())
            if not torch.isfinite(loss):raise RuntimeError(f'Invalid Retina loss: {parts}')
            scaler.scale(loss).backward();scaler.step(opt);scaler.update()
            n+=len(imgs)
            for k,v in parts.items():sums[k]+=float(v.detach())*len(imgs)
        record={'epoch':epoch,'train_loss':sum(sums.values())/max(n,1),
                'lr':opt.param_groups[0]['lr'],'train_min':(time.perf_counter()-t0)/60}
        if epoch%RETINA_VAL_EVERY==0 or epoch==RETINA_EPOCHS:
            preds,ms=retina_val_preds(net,val_selected,tta=False)
            metrics=coco_metrics(preds,COCO_CHECKPOINT)
            record.update({'val_'+k:v for k,v in metrics.items()});record['val_ms_per_image']=ms
            print('Retina epoch',epoch,'mAP',round(metrics['mAP'],4))
            if metrics['mAP']>best_map:
                best_map=metrics['mAP']
                atomic_save({'model':net.state_dict(),'epoch':epoch,'val_metrics':metrics,
                            'config_hash':hash_sig},RET_BEST)
        history.append(record);sch.step()
        atomic_save({'model':net.state_dict(),'optimizer':opt.state_dict(),
                     'scheduler':sch.state_dict(),'scaler':scaler.state_dict(),'epoch':epoch,
                     'best_map':best_map,'config_hash':hash_sig,'history':history},RET_LAST)
        pd.DataFrame(history).to_csv(OUT/'retina_history.csv',index=False)
    del net,opt,sch,scaler,loader;gc.collect();torch.cuda.empty_cache()
assert RET_BEST.exists(), 'No RetinaNet best checkpoint.'
print('Retina best checkpoint:',RET_BEST)

Retina best checkpoint: /kaggle/working/final_retina_rtdetr_pipeline/retina_best.pt


## 7. RT-DETR-L: COCO-pretrained, 50 denoising queries

Це окреме навчання, `num_denoising=50` перенесено з `RTDETR_denoising_tuning`.

In [17]:
from ultralytics.data.utils import img2label_paths
from ultralytics.models.rtdetr.train import RTDETRTrainer

class Denoising50Trainer(RTDETRTrainer):
    def get_model(self,cfg=None,weights=None,verbose=True):
        model=super().get_model(cfg,weights,verbose)
        head=model.model[-1]
        if not hasattr(head,'num_denoising'):
            raise RuntimeError('Ultralytics RTDETRDecoder API changed. Pin a compatible version.')
        print('RT-DETR denoising before:',head.num_denoising)
        head.num_denoising=RTDETR_DENOISING
        print('RT-DETR denoising after:',head.num_denoising)
        return model

def yolo_rel(rel):
    p=relative_safe(rel)
    if p.parts and p.parts[0]=='images':p=Path(*p.parts[1:])
    assert p.parts and 'images' not in p.parts
    return p

def yolo_label(rec):
    w,h=rec['width'],rec['height'];out=[]
    for b,c in zip(rec['boxes'],rec['labels']):
        x1,y1,x2,y2=map(float,b)
        if x2<=x1 or y2<=y1:continue
        vals=[(x1+x2)/(2*w),(y1+y2)/(2*h),(x2-x1)/w,(y2-y1)/h]
        vals=[min(1.,max(0.,float(z))) for z in vals]
        if vals[2]>0 and vals[3]>0:
            out.append(f'{int(c)-1} '+' '.join(f'{v:.7f}' for v in vals))
    return out

YOLO_ROOT=Path('/kaggle/working/visdrone_rtdetr_yolo')
YOLO_YAML=YOLO_ROOT/'data.yaml'
def build_yolo_data():
    settings={'layout':3,'train_step':TRAIN_FRAME_STEP,'val_step':VAL_FRAME_STEP,
              'train_ids_hash':hashlib.sha256(json.dumps([r['image_id'] for r in train_selected]).encode()).hexdigest(),
              'val_ids_hash':hashlib.sha256(json.dumps([r['image_id'] for r in val_selected]).encode()).hexdigest(),
              'data_root':str(DATA_ROOT)}
    marker=YOLO_ROOT/'.complete.json'
    if marker.is_file() and YOLO_YAML.is_file() and json.loads(marker.read_text())==settings:
        return YOLO_YAML
    if YOLO_ROOT.exists():shutil.rmtree(YOLO_ROOT)
    for split,rows,source_split in [('train',train_selected,'train'),('val',val_selected,'val')]:
        for r in tqdm(rows,desc=f'RT-DETR labels {split}'):
            relative=yolo_rel(r['file_name'])
            src=(DATA_ROOT/source_split/r['file_name']).resolve()
            assert src.is_file(),src
            img=YOLO_ROOT/'images'/split/relative
            label=YOLO_ROOT/'labels'/split/relative.with_suffix('.txt')
            img.parent.mkdir(parents=True,exist_ok=True)
            label.parent.mkdir(parents=True,exist_ok=True)
            img.symlink_to(src)
            label.write_text('\n'.join(yolo_label(r)),encoding='utf8')
            assert Path(img2label_paths([str(img)])[0])==label
    doc={'path':str(YOLO_ROOT),'train':'images/train','val':'images/val',
         'nc':8,'names':dict(enumerate(NAMES))}
    YOLO_YAML.write_text(yaml.safe_dump(doc,sort_keys=False))
    marker.write_text(json.dumps(settings,indent=2))
    return YOLO_YAML

RT_DIR=OUT/'rtdetr_run'
RT_BEST=RT_DIR/'weights'/'best.pt'
RT_LAST=RT_DIR/'weights'/'last.pt'
if RTDETR_BEST_INPUT:
    external=Path(RTDETR_BEST_INPUT)
    assert external.is_file(),external
    if not RT_BEST.exists():
        RT_BEST.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(external,RT_BEST)

if RTDETR_TRAIN:
    if RTDETR_BEST_INPUT:
        print('External RT-DETR best supplied; training disabled to avoid retraining.')
    else:
        yaml_path=build_yolo_data()
        RT_DIR.mkdir(exist_ok=True,parents=True)
        native_log=RT_DIR/'results.csv'
        already_complete=RT_BEST.is_file() and native_log.is_file() and len(pd.read_csv(native_log))>=RTDETR_EPOCHS
        if already_complete:
            print('RT-DETR already completed: skipping repeat training.')
        else:
            started=time.perf_counter()
            resume=RT_LAST.is_file()
            rt=RTDETR(str(RT_LAST) if resume else RTDETR_PRETRAINED)
            args=dict(data=str(yaml_path),epochs=RTDETR_EPOCHS,
                imgsz=RTDETR_IMGSZ,batch=RTDETR_BATCH,device=0,
                workers=NUM_WORKERS,seed=SEED,deterministic=False,
                project=str(OUT),name='rtdetr_run',exist_ok=True,
                optimizer='AdamW',lr0=RTDETR_LR,lrf=0.1,warmup_epochs=3.0,
                amp=RTDETR_AMP,patience=100,save=True,save_period=-1,
                val=True,plots=False,verbose=True,cache=False,
                mosaic=0.0,mixup=0.0,copy_paste=0.0,
                fliplr=0.5,flipud=0.0,degrees=0.0,translate=0.0,scale=0.0,
                trainer=Denoising50Trainer,resume=resume)
            if resume:
                # On resume Ultralytics restores hyperparams and epoch from last.pt.
                print('Resume RT-DETR from:',RT_LAST)
            rt.train(**args)
            (RT_DIR/'training_summary.json').write_text(json.dumps({
                'elapsed_minutes':(time.perf_counter()-started)/60,'resumed':resume,
                'epochs_requested':RTDETR_EPOCHS,'train_frame_step':TRAIN_FRAME_STEP,
                'denoising_queries':RTDETR_DENOISING,'imgsz':RTDETR_IMGSZ,
                'lr0':RTDETR_LR,'model':'RT-DETR-L'},indent=2))
            del rt;gc.collect();torch.cuda.empty_cache()
assert RT_BEST.exists(),f'RT-DETR best checkpoint not found: {RT_BEST}'
print('RT-DETR best:',RT_BEST)

RT-DETR labels train:   0%|          | 0/8090 [00:00<?, ?it/s]

RT-DETR labels val:   0%|          | 0/1425 [00:00<?, ?it/s]

Ultralytics 8.4.204 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/visdrone_rtdetr_yolo/data.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.001, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=rtdetr_run, nbs=64, nms=None, opset=None, 

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/10      2.14G     0.9799     0.5687     0.1183        113        640: 100% ━━━━━━━━━━━━ 4045/4045 4.6it/s 14:300.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 357/357 9.2it/s 38.7s<0.1ss
                   all       1425      57178      0.139      0.272      0.103      0.047

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       3/10      2.14G     0.6633     0.6399    0.07557         74        640: 100% ━━━━━━━━━━━━ 4045/4045 4.6it/s 14:330.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 357/357 9.1it/s 39.3s<0.2ss
                   all       1425      57178      0.423      0.312       0.26       0.14

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       4/10      2.14G     0.6137     0.5548     0.0716         81        640: 100% ━━━━━━━━━━━━ 4045/4045 4.6it/s 14:340.2ss
              

## 8. Validation

In [ ]:
VAL_RETINA_FILE=OUT/'val_retina.json.gz'
VAL_RETINA_TTA_FILE=OUT/'val_retina_tta.json.gz'
VAL_RT_FILE=OUT/'val_rtdetr.json.gz'
if not VAL_RETINA_FILE.exists() or (RETINA_TTA_ABLATION and not VAL_RETINA_TTA_FILE.exists()):
    net=make_retina_model(with_pretraining=False)
    load_retina_weights(net,RET_BEST);net.eval()
    if not VAL_RETINA_FILE.exists():
        vals,ms=retina_val_preds(net,val_final,tta=False)
        dump_gz(VAL_RETINA_FILE,vals)
        (OUT/'retina_speed.json').write_text(json.dumps({'ms_per_image':float(ms)}))
    if RETINA_TTA_ABLATION and not VAL_RETINA_TTA_FILE.exists():
        vals,ms=retina_val_preds(net,val_final,tta=True)
        dump_gz(VAL_RETINA_TTA_FILE,vals)
        (OUT/'retina_tta_speed.json').write_text(json.dumps({'ms_per_image':float(ms)}))
    del net;gc.collect();torch.cuda.empty_cache()

if not VAL_RT_FILE.exists():
    rt=RTDETR(str(RT_BEST))
    if hasattr(rt.model.model[-1],'num_denoising'):
        print('RT-DETR checkpoint num_denoising:',rt.model.model[-1].num_denoising)
    rows=[];duration=[]
    for i in tqdm(range(0,len(val_final),2),desc='RT-DETR full official val'):
        batch=val_final[i:i+2]
        paths=[str(DATA_ROOT/'val'/r['file_name']) for r in batch]
        t=time.perf_counter()
        results=rt.predict(source=paths,imgsz=RTDETR_IMGSZ,conf=EVAL_CONF,
                           max_det=300,device=0,quantize=16 if RTDETR_AMP else 32,verbose=False)
        torch.cuda.synchronize();duration.append((time.perf_counter()-t)/len(batch))
        assert len(results)==len(batch)
        for r,result in zip(batch,results):
            if result.boxes is None:continue
            rows += pred_from_xyxy(r['image_id'],result.boxes.xyxy.cpu().tolist(),
                (result.boxes.cls.long()+1).cpu().tolist(),result.boxes.conf.cpu().tolist())
    dump_gz(VAL_RT_FILE,rows)
    (OUT/'rtdetr_speed.json').write_text(json.dumps({'ms_per_image':float(1000*np.mean(duration))}))
    del rt;gc.collect();torch.cuda.empty_cache()

retina_val=read_gz(VAL_RETINA_FILE)
rt_val=read_gz(VAL_RT_FILE)
retina_tta_val=read_gz(VAL_RETINA_TTA_FILE) if RETINA_TTA_ABLATION else None
assert set(x['image_id'] for x in retina_val).issubset({r['image_id'] for r in val_final})
assert set(x['image_id'] for x in rt_val).issubset({r['image_id'] for r in val_final})
print('Prediction counts:',len(retina_val),len(rt_val))

Prediction counts: 486933 771335


## 9. Ensemble: class-aware NMS та Weighted Boxes Fusion (WBF)


In [ ]:
def fuse_predictions(rows_a,rows_b,records,method='nms',iou=0.55,weights=(0.7,0.3)):
    a=group_predictions(rows_a);b=group_predictions(rows_b)
    fused=[]
    for rec in tqdm(records,desc=f'Fusion {method} {weights} IoU={iou}',leave=False):
        iid = rec['image_id']
        key = str(iid)
        W = float(rec['width'])
        H = float(rec['height'])
        parts = [a.get(key, []), b.get(key, [])]
        if method=='nms':
            merged=parts[0]+parts[1]
            if not merged:continue
            boxes=torch.tensor([xywh_to_xyxy(x['bbox']) for x in merged],dtype=torch.float32)
            scores=torch.tensor([float(x['score'])*weights[j] for j,p in enumerate(parts) for x in p],dtype=torch.float32)
            labels=torch.tensor([int(x['category_id']) for x in merged],dtype=torch.long)
            # NMS scores use model weighting; use the weighted score for ranking and final output.
            keep=batched_nms(boxes,scores,labels,iou)[:MAX_DETECTIONS]
            fused+=pred_from_xyxy(iid,boxes[keep].tolist(),labels[keep].tolist(),scores[keep].tolist())
        elif method=='wbf':
            box_lists=[];score_lists=[];label_lists=[]
            for p in parts:
                box_lists.append([[min(1.,max(0.,z/W)),min(1.,max(0.,y/H)),
                                   min(1.,max(0.,v/W)),min(1.,max(0.,t/H))]
                     for x in p for z,y,v,t in [xywh_to_xyxy(x['bbox'])]])
                score_lists.append([float(x['score']) for x in p])
                label_lists.append([int(x['category_id']) for x in p])
            if not any(score_lists):continue
            fb,fs,fl=weighted_boxes_fusion(box_lists,score_lists,label_lists,
                weights=list(weights),iou_thr=iou,skip_box_thr=EVAL_CONF,conf_type='avg')
            boxes=[[float(bb[0])*W,float(bb[1])*H,float(bb[2])*W,float(bb[3])*H] for bb in fb]
            labels=[int(z) for z in fl];scores=[float(s) for s in fs]
            order=np.argsort(scores)[::-1][:MAX_DETECTIONS]
            fused+=pred_from_xyxy(iid,[boxes[j] for j in order],[labels[j] for j in order],
                                  [scores[j] for j in order])
        else:raise ValueError(method)
    return fused

val_inputs={'retina':retina_val,'rtdetr':rt_val}
if retina_tta_val is not None:val_inputs['retina_tta']=retina_tta_val
metrics={name:coco_metrics(preds) for name,preds in val_inputs.items()}
retina_variant=max(('retina','retina_tta') if retina_tta_val is not None else ('retina',),
                   key=lambda x:metrics[x]['mAP'])
print('Best Retina inference:',retina_variant)

FUSION_SPECS={
    'nms_r70_t30':dict(method='nms',iou=0.55,weights=(0.7,0.3)),
    'wbf_r70_t30':dict(method='wbf',iou=0.55,weights=(0.7,0.3)),
    'wbf_r50_t50':dict(method='wbf',iou=0.50,weights=(0.5,0.5)),
}
fused_outputs={}
for name,spec in FUSION_SPECS.items():
    fused=fuse_predictions(val_inputs[retina_variant],rt_val,val_final,**spec)
    fused_outputs[name]=fused
    metrics[name]=coco_metrics(fused)
    dump_gz(OUT/f'val_{name}.json.gz',fused)

metrics_df=pd.DataFrame(metrics).T.sort_values('mAP',ascending=False)
metrics_df.to_csv(OUT/'val_comparison.csv',index_label='candidate')
print(metrics_df[['mAP','AP50','AP_small','AP_medium','AP_large']].round(5).to_string())
WINNER=str(metrics_df.index[0])
selection={'chosen':WINNER,'best_retina_variant':retina_variant,
           'ranking_by':'official_val_COCO_mAP','metrics':metrics,
           'fusion_specs':FUSION_SPECS,'seed':SEED,'val_frame_step':FINAL_VAL_FRAME_STEP,
           'n_val_images':len(val_final),'test_score_threshold':EVAL_CONF}
(OUT/'ensemble_selection.json').write_text(json.dumps(selection,indent=2,ensure_ascii=False))
print('SELECTED:',WINNER)

Best Retina inference: retina_tta


Fusion nms (0.7, 0.3) IoU=0.55:   0%|          | 0/2846 [00:00<?, ?it/s]

Fusion wbf (0.7, 0.3) IoU=0.55:   0%|          | 0/2846 [00:00<?, ?it/s]

Fusion wbf (0.5, 0.5) IoU=0.5:   0%|          | 0/2846 [00:00<?, ?it/s]

                 mAP     AP50  AP_small  AP_medium  AP_large
nms_r70_t30  0.16908  0.33788   0.07333    0.16058   0.30781
wbf_r50_t50  0.16805  0.34103   0.08253    0.15638   0.30681
retina_tta   0.16166  0.31763   0.06834    0.15186   0.30684
rtdetr       0.16082  0.31723   0.08235    0.14983   0.28959
wbf_r70_t30  0.15805  0.32000   0.07138    0.14742   0.29725
retina       0.15185  0.29802   0.05579    0.14097   0.29375
SELECTED: nms_r70_t30


## 10. Test inference


In [ ]:
def prepare_test_rows():
    rows=[]
    for rec in test_info.to_dict('records'):
        row=dict(rec)
        row['path']=str(locate_image('test',rec['file_name']))
        row['image_id']=str(rec['image_id'])
        rows.append(row)
    assert len(rows)==len(test_info)
    return rows

TEST_ROWS=prepare_test_rows() if RUN_TEST else []
TEST_RET_FILE=OUT/'test_retina.json.gz'
TEST_RET_TTA_FILE=OUT/'test_retina_tta.json.gz'
TEST_RT_FILE=OUT/'test_rtdetr.json.gz'

def retina_test_predictions(weights_path,do_tta):
    net=make_retina_model(False);load_retina_weights(net,weights_path)
    base=[];tta=[]
    for r in tqdm(TEST_ROWS,desc='Retina test + optional TTA'):
        with Image.open(r['path']) as im: rgb=np.asarray(im.convert('RGB')).copy()
        h,w=rgb.shape[:2]
        assert (w,h)==(int(r['width']),int(r['height']))
        x=torch.from_numpy(rgb).permute(2,0,1).float()/255
        # Save baseline separately even if TTA is selected.
        p=retina_infer_one(net,x,False)
        base_output=p
        base+=pred_from_xyxy(r['image_id'],p['boxes'].tolist(),p['labels'].tolist(),p['scores'].tolist())
        if do_tta:
            p=retina_infer_one(net,x,True,base=base_output)
            tta+=pred_from_xyxy(r['image_id'],p['boxes'].tolist(),p['labels'].tolist(),p['scores'].tolist())
    del net;gc.collect();torch.cuda.empty_cache()
    return base,tta

if RUN_TEST:
    need_tta=(retina_variant=='retina_tta')
    if not TEST_RET_FILE.exists() or (need_tta and not TEST_RET_TTA_FILE.exists()):
        baseline,tta=retina_test_predictions(RET_BEST,need_tta)
        dump_gz(TEST_RET_FILE,baseline)
        if need_tta:dump_gz(TEST_RET_TTA_FILE,tta)
    if not TEST_RT_FILE.exists():
        rt=RTDETR(str(RT_BEST))
        rt_rows=[]
        for k in tqdm(range(0,len(TEST_ROWS),2),desc='RT-DETR test'):
            group=TEST_ROWS[k:k+2]
            results=rt.predict(source=[x['path'] for x in group],imgsz=RTDETR_IMGSZ,
                conf=EVAL_CONF,max_det=300,device=0,half=RTDETR_AMP,verbose=False)
            assert len(results)==len(group)
            for rec,result in zip(group,results):
                if result.boxes is None:continue
                rt_rows+=pred_from_xyxy(rec['image_id'],result.boxes.xyxy.cpu().tolist(),
                    (result.boxes.cls.long()+1).cpu().tolist(),result.boxes.conf.cpu().tolist())
        dump_gz(TEST_RT_FILE,rt_rows)
        del rt;gc.collect();torch.cuda.empty_cache()
    test_retina=read_gz(TEST_RET_FILE)
    test_rt=read_gz(TEST_RT_FILE)
    test_retina_selected=read_gz(TEST_RET_TTA_FILE) if need_tta else test_retina
    print('Test raw detection totals: Retina',len(test_retina),'RTDETR',len(test_rt))

    def test_records_for_fusion():
        return [{'image_id':x['image_id'],'width':int(x['width']),'height':int(x['height'])}
                for x in TEST_ROWS]

    all_test={'retina':test_retina,'rtdetr':test_rt}
    if need_tta:all_test['retina_tta']=test_retina_selected
    for name,spec in FUSION_SPECS.items():
        savefile=OUT/f'test_{name}.json.gz'
        if not savefile.exists():
            fp=fuse_predictions(test_retina_selected,test_rt,test_records_for_fusion(),**spec)
            dump_gz(savefile,fp)
        all_test[name]=read_gz(savefile)
    if WINNER not in all_test:raise ValueError(f'Chosen output {WINNER} missing')

    def save_submission(preds,path):
        byid=group_predictions(preds)
        dimension=test_info.set_index('image_id')[['width','height']].to_dict('index')
        strings={}
        for iid,p in byid.items():
            assert iid in dimension,iid
            w,h=float(dimension[iid]['width']),float(dimension[iid]['height'])
            chunks=[]
            for item in sorted(p,key=lambda x:x['score'],reverse=True)[:MAX_DETECTIONS]:
                cid=int(item['category_id'])-1
                x,y,bw,bh=map(float,item['bbox'])
                x=max(0.,min(x,w));y=max(0.,min(y,h))
                bw=max(0.,min(bw,w-x));bh=max(0.,min(bh,h-y))
                score=float(item['score'])
                if not(0<=cid<=7 and 0<=score<=1):raise ValueError('Invalid class or score')
                if bw<=0 or bh<=0:continue
                chunks.append(f'{cid} {score:.6f} {x:.2f} {y:.2f} {bw:.2f} {bh:.2f}')
            strings[iid]=' '.join(chunks)
        df=sample[['image_id']].copy()
        df['PredictionString']=df.image_id.map(strings).fillna('')
        assert len(df)==len(test_info) and df.image_id.is_unique
        assert df.PredictionString.notna().all()
        assert all(len(x.split())%6==0 for x in df.PredictionString)
        df.to_csv(path,index=False)
        print('Submission:',path,'| rows',len(df),'| nonempty',(df.PredictionString!='').sum())
        return df

    for key,preds in all_test.items():
        save_submission(preds,OUT/f'submission_{key}.csv')
    for rank,key in enumerate(metrics_df.index[:2],start=1):
        save_submission(all_test[str(key)],OUT/f'submission_candidate_{rank}.csv')
    chosen_df=save_submission(all_test[WINNER],Path('/kaggle/working/submission.csv'))
    display(chosen_df.head(3))
else:
    print('RUN_TEST=False — saved model/checkpoint and validation scores only.')

Retina test + optional TTA:   0%|          | 0/3811 [00:00<?, ?it/s]

RT-DETR test:   0%|          | 0/1906 [00:00<?, ?it/s]

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.

Fusion nms (0.7, 0.3) IoU=0.55:   0%|          | 0/3811 [00:00<?, ?it/s]

Fusion wbf (0.7, 0.3) IoU=0.55:   0%|          | 0/3811 [00:00<?, ?it/s]

Fusion wbf (0.5, 0.5) IoU=0.5:   0%|          | 0/3811 [00:00<?, ?it/s]

Submission: /kaggle/working/final_retina_rtdetr_pipeline/submission_retina.csv | rows 3811 | nonempty 3811
Submission: /kaggle/working/final_retina_rtdetr_pipeline/submission_rtdetr.csv | rows 3811 | nonempty 3811
Submission: /kaggle/working/final_retina_rtdetr_pipeline/submission_retina_tta.csv | rows 3811 | nonempty 3811
Submission: /kaggle/working/final_retina_rtdetr_pipeline/submission_nms_r70_t30.csv | rows 3811 | nonempty 3811
Submission: /kaggle/working/final_retina_rtdetr_pipeline/submission_wbf_r70_t30.csv | rows 3811 | nonempty 3811
Submission: /kaggle/working/final_retina_rtdetr_pipeline/submission_wbf_r50_t50.csv | rows 3811 | nonempty 3811
Submission: /kaggle/working/final_retina_rtdetr_pipeline/submission_candidate_1.csv | rows 3811 | nonempty 3811
Submission: /kaggle/working/final_retina_rtdetr_pipeline/submission_candidate_2.csv | rows 3811 | nonempty 3811
Submission: /kaggle/working/submission.csv | rows 3811 | nonempty 3811


,image_id,PredictionString
0,clip_001_0000001,0 0.687695 976.72 716.82 35.70 94.81 0 0.67094...
1,clip_001_0000002,0 0.688037 976.09 718.55 35.52 94.23 0 0.68051...
2,clip_001_0000003,0 0.690430 974.44 720.05 35.94 97.00 0 0.67983...
